In [3]:
import os, urllib.request
if not os.path.exists("utils.py"):
    urllib.request.urlretrieve(
        "https://raw.githubusercontent.com/abgoswam/swe_in_prod_vizuara_01/main/utils.py",
        "utils.py"
    )

import json, random
import numpy as np
import pandas as pd
import torch
from IPython.display import display

from utils import (TARGET, MOCK_FILE, MockEnv, SYSTEM, NO_COMMAND, first_bash_block, generate, scripted_sampler, show_rollout, show_group)

for _opt in ["display.max_colwidth", "display.max_rows", "display.max_columns", "display.width"]:
    pd.set_option(_opt, None)

random.seed(0); torch.manual_seed(0)
DEV = "cuda" if torch.cuda.is_available() else "cpu"
print("device: ", DEV)

device:  cuda


## Load one instance

In [4]:
from datasets import load_dataset

ds = load_dataset("princeton-nlp/SWE-bench_Verified", split="test") 
print(ds)

# A small single-file task keeps the walkthrough readable.
cands = [i for i, r in enumerate(ds)
            if r["patch"].count("diff --git") == 1 and len(r["patch"]) < 1800]

inst = ds[cands[0]] 
fail_to_pass = json.loads(inst["FAIL_TO_PASS"])
pass_to_pass = json.loads(inst["PASS_TO_PASS"])

display(pd.DataFrame(
    [(k, inst.get(k)) for k in ["instance_id", "repo", "base_commit", "version", "difficulty"]],
    columns=["field", "value"],
))

display(pd.DataFrame([
    ("1.1", "problem_statement", "the agent",  "the input: a raw GitHub issue"),
    ("-",   "patch",             "nobody",     "reference solution; unused in this notebook"),
    ("1.2", "test_patch",        "the grader", "adds the tests that define 'fixed'"),
    ("1.3", "FAIL_TO_PASS",      "the grader", "must go red -> green"),
    ("1.4", "PASS_TO_PASS",      "the grader", "must stay green"),
], columns=["section", "field", "who sees it", "job"]))

README.md:   0%|          | 0.00/3.34k [00:00<?, ?B/s]

data/test-00000-of-00001.parquet: reconstructing file:   0%|          |  0.00B / 2.10MB            

data/test-00000-of-00001.parquet: downloading bytes:           |  0.00B            

Generating test split:   0%|          | 0/500 [00:00<?, ? examples/s]

Dataset({
    features: ['repo', 'instance_id', 'base_commit', 'patch', 'test_patch', 'problem_statement', 'hints_text', 'created_at', 'version', 'FAIL_TO_PASS', 'PASS_TO_PASS', 'environment_setup_commit', 'difficulty'],
    num_rows: 500
})


,field,value
0,instance_id,astropy__astropy-12907
1,repo,astropy/astropy
2,base_commit,d16bfe05a744909de4b27f5875fe0d4ed41ce607
3,version,4.3
4,difficulty,15 min - 1 hour


,section,field,who sees it,job
0,1.1,problem_statement,the agent,the input: a raw GitHub issue
1,-,patch,nobody,reference solution; unused in this notebook
2,1.2,test_patch,the grader,adds the tests that define 'fixed'
3,1.3,FAIL_TO_PASS,the grader,must go red -> green
4,1.4,PASS_TO_PASS,the grader,must stay green


## 1.1 problem_statement — the agent's entire input
#### A raw GitHub issue. The agent is never told which file to open, or that separable.py exists. Finding it — localization — is most of the real difficulty of SWE-bench.

In [5]:
print(inst["problem_statement"])

Modeling's `separability_matrix` does not compute separability correctly for nested CompoundModels
Consider the following model:

```python
from astropy.modeling import models as m
from astropy.modeling.separable import separability_matrix

cm = m.Linear1D(10) & m.Linear1D(5)
```

It's separability matrix as you might expect is a diagonal:

```python
>>> separability_matrix(cm)
array([[ True, False],
       [False,  True]])
```

If I make the model more complex:
```python
>>> separability_matrix(m.Pix2Sky_TAN() & m.Linear1D(10) & m.Linear1D(5))
array([[ True,  True, False, False],
       [ True,  True, False, False],
       [False, False,  True, False],
       [False, False, False,  True]])
```

The output matrix is again, as expected, the outputs and inputs to the linear models are separable and independent of each other.

If however, I nest these compound models:
```python
>>> separability_matrix(m.Pix2Sky_TAN() & cm)
array([[ True,  True, False, False],
       [ True,  True, False, 

In [6]:
print(inst["test_patch"])

diff --git a/astropy/modeling/tests/test_separable.py b/astropy/modeling/tests/test_separable.py
--- a/astropy/modeling/tests/test_separable.py
+++ b/astropy/modeling/tests/test_separable.py
@@ -28,6 +28,13 @@
 p1 = models.Polynomial1D(1, name='p1')
 
 
+cm_4d_expected = (np.array([False, False, True, True]),
+                  np.array([[True,  True,  False, False],
+                            [True,  True,  False, False],
+                            [False, False, True,  False],
+                            [False, False, False, True]]))
+
+
 compound_models = {
     'cm1': (map3 & sh1 | rot & sh1 | sh1 & sh2 & sh1,
             (np.array([False, False, True]),
@@ -52,7 +59,17 @@
     'cm7': (map2 | p2 & sh1,
             (np.array([False, True]),
              np.array([[True, False], [False, True]]))
-            )
+            ),
+    'cm8': (rot & (sh1 & sh2), cm_4d_expected),
+    'cm9': (rot & sh1 & sh2, cm_4d_expected),
+    'cm10': ((rot & sh1) & sh2, cm_4d_expected),
+    

## FAIL_TO_PASS - must go red -> green
#### The half of the reward that says you solved the issue.

In [7]:
for t in fail_to_pass:
    print(t)

astropy/modeling/tests/test_separable.py::test_separable[compound_model6-result6]
astropy/modeling/tests/test_separable.py::test_separable[compound_model9-result9]
